<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/DL-2026/Lecture_3/Lecture_3_Convolutional_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Глава 1. Пример 1: один вход, одно ядро (свёртка 1D)

Прежде чем строить глубокие свёрточные сети, полезно разобрать простейшую операцию, в которой виден каждый шаг вычислений. Эта глава — «эталонный пример» для свёрток: на нём удобно проверить, что вы понимаете прямой проход, цепное правило с разделением весов (weight sharing) и правило обновления. Дальше, в следующих главах, мы будем ссылаться на формулы из этой главы как на базовые. К каждой части главы прилагается код на Python, который порождает соответствующую схему.

## 1.1. Зачем нужен этот пример

Одномерная свёртка с одним ядром — это, по сути, композиция скользящего скалярного произведения и нелинейности:

$$
z_i = b + \sum_{m=1}^{K} w_m \, x_{i+m-1}, \qquad a_i = \sigma(z_i).
$$

В этом примере всего четыре обучаемых параметра: три веса ядра $w_1, w_2, w_3$ и одно смещение $b$. Их так мало, что все производные можно выписать в явном виде и проследить каждую стрелку на графе вычислений. Именно поэтому этот пример хорош как первая иллюстрация:

- **прямой проход** здесь — это три применения скалярного произведения;
- **обратный проход** — это суммирование градиентов по всем позициям, где побывало ядро;
- **обновление** — четыре вычитания (так как параметров четыре).

Разобравшись в этом, вы без труда перенесёте ту же логику на двумерные свёртки с сотнями каналов: изменится число стрелок и размерность, но структура вычислений останется той же.

## 1.2. Архитектура и обозначения

Сеть (точнее, один свёрточный слой) имеет **один входной вектор**, **одно ядро** и **один выходной вектор**. Обозначим:

| Символ | Смысл |
|---|---|
| $x$ | входной сигнал, вектор длины $n$ |
| $w_m$ | вес ядра (фильтра), $m=1,\dots,K$ |
| $b$ | смещение (bias), одно число |
| $K$ | длина ядра (размер окна) |
| $z_i$ | преактивация $i$-го выхода |
| $a_i$ | активация $i$-го выхода |
| $\sigma$ | функция активации (в примерах — сигмоида) |
| $t_i$ | истинный ответ |
| $L$ | функция потерь |
| $\eta$ | шаг обучения (learning rate) |

Итого **четыре обучаемых параметра**: $w_1$, $w_2$, $w_3$, $b$. На рис. 1.1 изображена соответствующая схема: входные синие блоки, скользящее желтое ядро, красная рамка текущего окна и серые блоки выходов.



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch


def draw_architecture():
    """Рис. 1.1: архитектура свёртки 1D (один вход, одно ядро)."""
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 7)
    ax.axis('off')

    # 1. Входной вектор (x)
    x_vals = ['x₁', 'x₂', 'x₃', 'x₄', 'x₅']
    for i, val in enumerate(x_vals):
        rect = patches.Rectangle((1 + i * 1.2, 2), 1, 1,
                                 facecolor='#AEDFF7', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i * 1.2, 2.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    ax.text(3.4, 3.6, r'Входной сигнал $x$',
            ha='center', fontsize=12, style='italic')

    # 2. Ядро (w)
    for i, val in enumerate(['w₁', 'w₂', 'w₃']):
        rect = patches.Rectangle((1 + i * 1.2, 4.5), 1, 1,
                                 facecolor='#FFE29A', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i * 1.2, 5.0, val, ha='center', va='center',
                fontsize=14, zorder=4)

    ax.text(2.7, 6.0, r'Ядро (фильтр) $w$',
            ha='center', fontsize=12, style='italic')

    # Стрелки от ядра к первому окну
    for i in range(3):
        ax.annotate('', xy=(1.5 + i * 1.2, 3.05), xytext=(1.5 + i * 1.2, 4.45),
                    arrowprops=dict(arrowstyle='->', color='#B22222', lw=2),
                    zorder=2)

    # Рамка вокруг первого окна
    window_rect = patches.Rectangle((0.8, 1.7), 3.6, 1.6, fill=False,
                                    edgecolor='#E74C3C', linestyle='--',
                                    lw=2.5, zorder=5)
    ax.add_patch(window_rect)
    ax.text(2.6, 1.3, 'Окно 1', ha='center', fontsize=11,
            color='#E74C3C', fontweight='bold')

    # 3. Выход (z)
    for i, val in enumerate(['z₁', 'z₂', 'z₃']):
        rect = patches.Rectangle((9 + i * 1.5, 2), 1, 1,
                                 facecolor='#E8E8E8', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(9.5 + i * 1.5, 2.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    ax.text(10.5, 3.5, r'Выход $z$ (длина 3)',
            ha='center', fontsize=12, style='italic')

    # Стрелка от окна к z1 (с изгибом)
    arrow = FancyArrowPatch((4.4, 2.0), (8.9, 2.5),
                            arrowstyle='-|>', mutation_scale=20, lw=2.5,
                            color='#2C3E50',
                            connectionstyle='arc3,rad=-0.2', zorder=2)
    ax.add_patch(arrow)

    ax.text(6.6, 1.5, 'Скалярное\nпроизведение',
            ha='center', fontsize=11, color='#2C3E50')

    # Смещение (b)
    ax.annotate('', xy=(9.5, 1.95), xytext=(9.5, 0.5),
                arrowprops=dict(arrowstyle='->', color='#1F6FB2', lw=2.5),
                zorder=2)
    ax.text(9.8, 1.0, r'Смещение $b$',
            fontsize=12, color='#1F6FB2', fontweight='bold')

    # 4. Активация
    rect = patches.Rectangle((13.5, 1.0), 1.2, 3.0,
                             facecolor='#B8E0B8', edgecolor='black',
                             lw=1.5, zorder=3)
    ax.add_patch(rect)
    ax.text(14.1, 2.5, 'Активация\n' + r'$\sigma(z)$',
            ha='center', va='center', fontsize=12, zorder=4)

    # Стрелка от z3 к активации
    ax.annotate('', xy=(13.45, 2.5), xytext=(12.5, 2.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2),
                zorder=2)

    ax.text(7.5, 0.0,
            r'Рис. 1.1. Свёртка 1D: окно из 3 элементов и ядро дают первый выход $z_1$',
            ha='center', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()


draw_architecture()


## 1.3. Прямой проход

Прямой проход (**forward pass**) — это вычисление выхода свёртки по заданному входу $x$. Оно выполняется строго последовательно, от входа к выходу; каждый следующий шаг опирается на результат предыдущего.

### Шаг 1. Линейная комбинация (преактивация)

Ядро «накладывается» на первые три элемента входа и вычисляет взвешенную сумму со смещением:

$$
z_1 = b + w_1 x_1 + w_2 x_2 + w_3 x_3.
$$

Здесь $z_1$ — это **преактивация**: значение до применения нелинейности. Затем окно сдвигается на одну позицию вправо (по умолчанию шаг $s=1$), и вычисляются остальные выходы. В общем виде для $i$-го выхода:

$$
z_i = b + \sum_{m=1}^{K} w_m \, x_{i+m-1}, \qquad i = 1, \dots, o,
$$

где длина выхода $o = n - K + 1$. Для нашего примера $n=5$, $K=3$, значит $o=3$:

$$
z_1 = b + w_1 x_1 + w_2 x_2 + w_3 x_3,
$$
$$
z_2 = b + w_1 x_2 + w_2 x_3 + w_3 x_4,
$$
$$
z_3 = b + w_1 x_3 + w_2 x_4 + w_3 x_5.
$$

Обратите внимание: веса $w_1, w_2, w_3$ используются во всех трех окнах. Это и есть **разделение весов** (weight sharing).

### Шаг 2. Активация

К каждой преактивации применяется нелинейная функция активации:

$$
a_i = \sigma(z_i).
$$

Для сигмоиды $\sigma(z) = \frac{1}{1 + e^{-z}}$. Величина $a_i$ — это **выход слоя**, который пойдёт дальше (например, на следующий свёрточный слой или в полносвязный слой).

### Схема прямого прохода

На рис. 1.2 те же шаги подписаны прямо над стрелками: слева направо вы видите, как окно вырезает часть входа, умножает на ядро, складывает со смещением и применяет активацию.




In [ ]:
def draw_forward_pass():
    """Рис. 1.2: архитектура + шаги прямого прохода."""
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 8)
    ax.axis('off')

    # Вход
    for i, val in enumerate(['$x_1$', '$x_2$', '$x_3$', '$x_4$', '$x_5$']):
        rect = patches.Rectangle((1 + i * 1.2, 3), 1, 1,
                                 facecolor='#AEDFF7', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i * 1.2, 3.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    # Ядро
    for i, val in enumerate(['$w_1$', '$w_2$', '$w_3$']):
        rect = patches.Rectangle((1 + i * 1.2, 5.5), 1, 1,
                                 facecolor='#FFE29A', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i * 1.2, 6.0, val, ha='center', va='center',
                fontsize=14, zorder=4)

    # Стрелки ядра
    for i in range(3):
        ax.annotate('', xy=(1.5 + i * 1.2, 4.05), xytext=(1.5 + i * 1.2, 5.45),
                    arrowprops=dict(arrowstyle='->', color='#B22222', lw=2),
                    zorder=2)

    # Рамка окна
    window_rect = patches.Rectangle((0.8, 2.7), 3.6, 1.6, fill=False,
                                    edgecolor='#E74C3C', linestyle='--',
                                    lw=2.5, zorder=5)
    ax.add_patch(window_rect)

    # Выход z
    for i, val in enumerate(['$z_1$', '$z_2$', '$z_3$']):
        rect = patches.Rectangle((9 + i * 1.5, 3), 1, 1,
                                 facecolor='#E8E8E8', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(9.5 + i * 1.5, 3.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    # Стрелка к z1
    arrow = FancyArrowPatch((4.4, 3.0), (8.9, 3.5),
                            arrowstyle='-|>', mutation_scale=20, lw=2.5,
                            color='#2C3E50',
                            connectionstyle='arc3,rad=-0.2', zorder=2)
    ax.add_patch(arrow)

    # Активация
    rect = patches.Rectangle((13.5, 2.0), 1.2, 3.0,
                             facecolor='#B8E0B8', edgecolor='black',
                             lw=1.5, zorder=3)
    ax.add_patch(rect)
    ax.text(14.1, 3.5, 'Активация\n' + r'$\sigma(z)$',
            ha='center', va='center', fontsize=12, zorder=4)
    ax.annotate('', xy=(13.45, 3.5), xytext=(12.5, 3.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2),
                zorder=2)

    # Формулы
    ax.text(4.5, 1.8, '$z_1 = b + w_1x_1 + w_2x_2 + w_3x_3$',
            ha='center', fontsize=12, color='#2C3E50')
    ax.text(10.5, 1.8, r'$a_1 = \sigma(z_1)$',
            ha='center', fontsize=12, color='#2C3E50')

    ax.text(8.0, 7.5,
            'Рис. 1.2. Прямой проход: скалярное произведение и активация',
            ha='center', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()


draw_forward_pass()



## 1.4. Функция потерь

После прямого прохода у нас есть предсказания $a_1, a_2, a_3$. Сравним их с истинными ответами $t_1, t_2, t_3$ и введём функцию потерь — half mean squared error:

$$
L = \frac{1}{2} \sum_{i=1}^{o} (a_i - t_i)^2.
$$

Множитель $1/2$ введён исключительно для удобства: при дифференцировании он сокращается с двойкой из производной квадрата, и в результате получается простая формула $\partial L/\partial a_i = a_i - t_i$ (см. шаг 1 обратного прохода).

## 1.5. Обратный проход

Обратный проход (**backward pass**, или backpropagation) — это вычисление производных $L$ по всем параметрам. Мы применяем цепное правило, шаг за шагом, двигаясь **справа налево** по графу вычислений прямого прохода. Главное отличие от полносвязного слоя здесь в том, что каждый вес $w_m$ используется во всех окнах. Поэтому при обратном проходе градиенты **суммируются по всем позициям**.

### Шаг 1. Локальный градиент (дельта) для каждой позиции

Дифференцируем $L$ по выходу $a_i$, а затем по преактивации $z_i$:

$$
\frac{\partial L}{\partial a_i} = a_i - t_i, \qquad \frac{\partial a_i}{\partial z_i} = \sigma'(z_i).
$$

Вводим обозначение для «дельты» $i$-й позиции:

$$
\delta_i := \frac{\partial L}{\partial z_i} = \frac{\partial L}{\partial a_i} \cdot \frac{\partial a_i}{\partial z_i} = (a_i - t_i)\,\sigma'(z_i).
$$

Для сигмоиды $\sigma'(z) = \sigma(z)(1 - \sigma(z))$. С этого момента $\delta_i$ — это удобное сокращение; оно полностью вычисляется из величин прямого прохода.

### Шаг 2. Градиент по смещению $b$

Смещение $b$ прибавляется к **каждой** преактивации $z_i$. По правилу суммы производных:

$$
\frac{\partial L}{\partial b} = \sum_{i=1}^{o} \frac{\partial L}{\partial z_i} \cdot \frac{\partial z_i}{\partial b}.
$$

Поскольку $z_i = b + \sum w_m x_{i+m-1}$, производная $\partial z_i/\partial b = 1$. Тогда:

$$
\frac{\partial L}{\partial b} = \sum_{i=1}^{o} \delta_i \cdot 1 = \delta_1 + \delta_2 + \delta_3.
$$

### Шаг 3. Градиенты по ядру $w$

Каждый вес $w_m$ участвует в каждом окне. Для примера выпишем **полную цепочку** для $w_1$:

$$
\frac{\partial L}{\partial w_1} = \sum_{i=1}^{o} \frac{\partial L}{\partial z_i} \cdot \frac{\partial z_i}{\partial w_1}.
$$

Из формулы $z_i$ видно, что $w_1$ всегда умножается на первый элемент $i$-го окна, то есть на $x_i$. Поэтому $\partial z_i/\partial w_1 = x_i$. Подставляем:

$$
\frac{\partial L}{\partial w_1} = \delta_1 x_1 + \delta_2 x_2 + \delta_3 x_3.
$$

Аналогично для $w_2$ (умножается на $x_{i+1}$) и $w_3$ (умножается на $x_{i+2}$):

$$
\frac{\partial L}{\partial w_2} = \delta_1 x_2 + \delta_2 x_3 + \delta_3 x_4,
$$
$$
\frac{\partial L}{\partial w_3} = \delta_1 x_3 + \delta_2 x_4 + \delta_3 x_5.
$$

В матричном виде это можно записать как $X^T \delta$, где $X$ — матрица из сдвинутых окон.

### Схема обратного прохода

Рис. 1.3 объединяет обе фазы: сверху сплошными синими стрелками идёт прямой проход, снизу пунктирными красными — обратный. От $\delta_1, \delta_2, \delta_3$ вниз отходят формулы для градиентов.




In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch


def draw_backward_pass():
    """Рис. 1.3: прямой (синие →) и обратный (красные ⇢) проход для свёртки."""
    fig, ax = plt.subplots(figsize=(15, 8))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 9)
    ax.axis('off')

    ax.text(8.0, 8.5,
            'Рис. 1.3. Прямой (синие →) и обратный (красные ⇢) проход для свёртки',
            ha='center', fontsize=14, fontweight='bold')

    # --- 1. ЯДРО (w) — СВЕРХУ ---
    for i, val in enumerate(['$w_1$', '$w_2$', '$w_3$']):
        rect = patches.Rectangle((2 + i * 1.5, 6.5), 1, 1,
                                 facecolor='#FFE29A', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(2.5 + i * 1.5, 7.0, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(3.5, 7.8, 'Ядро (фильтр) $w$',
            ha='center', fontsize=12, style='italic')

    # --- 2. ВХОД (x) — В ЦЕНТРЕ ---
    for i, val in enumerate(['$x_1$', '$x_2$', '$x_3$', '$x_4$', '$x_5$']):
        rect = patches.Rectangle((1 + i * 1.3, 4), 1, 1,
                                 facecolor='#AEDFF7', edgecolor='black',
                                 lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(1.5 + i * 1.3, 4.5, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(3.5, 3.3, 'Входной сигнал $x$',
            ha='center', fontsize=12, style='italic')

    # Прямой проход (синие сплошные): w → x
    for i in range(3):
        ax.annotate('', xy=(2.5 + i * 1.5, 5.05), xytext=(2.5 + i * 1.5, 6.45),
                    arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2),
                    zorder=2)

    # Рамка первого окна (x1, x2, x3) — раньше забывали add_patch
    window_rect = patches.Rectangle((0.8, 3.7), 4.1, 1.6, fill=False,
                                    edgecolor='#2C3E50', linestyle='--',
                                    lw=2, zorder=5)
    ax.add_patch(window_rect)

    # --- 3. ДЕЛЬТЫ (δ) — СНИЗУ ---
    d_positions = [(2.5, 1.5), (4.5, 1.5), (6.5, 1.5)]
    for i, (cx, cy) in enumerate(d_positions):
        ax.add_patch(patches.Circle((cx, cy), 0.45,
                                    facecolor='#FADBD8', edgecolor='#E74C3C',
                                    lw=2, zorder=3))
        # ВАЖНО: {i+1} без удвоения — иначе подставится literal "{i+1}"
        ax.text(cx, cy, rf'$\delta_{i+1}$',
                ha='center', va='center', fontsize=14,
                color='#C0392B', zorder=4)

    ax.text(4.5, 0.6, r'Ошибки (дельты) $\delta$',
            ha='center', fontsize=12, style='italic', color='#C0392B')

    # Прямой проход (синие сплошные): x → δ
    for cx in (2.5, 4.5, 6.5):
        ax.annotate('', xy=(cx, 2.05), xytext=(cx, 3.95),
                    arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2),
                    zorder=2)

    # --- 4. ОБРАТНЫЙ ПРОХОД (красные пунктирные): δ → w ---
    backward = [
        ((2.1, 1.5), (2.1, 6.4),  0.2),
        ((2.5, 1.9), (3.1, 6.4),  0.2),
        ((4.5, 1.9), (3.5, 6.4),  0.0),
        ((4.5, 1.9), (4.5, 6.4),  0.0),
        ((6.5, 1.9), (4.9, 6.4), -0.2),
        ((6.9, 1.5), (5.9, 6.4), -0.2),
    ]
    for (x0, y0), (x1, y1), rad in backward:
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1),
                                     arrowstyle='-|>', mutation_scale=15,
                                     lw=1.5, color='#E74C3C',
                                     linestyle='--',
                                     connectionstyle=f'arc3,rad={rad}'))

    # --- 5. ФОРМУЛЫ ГРАДИЕНТОВ (справа) ---
    formula_x = 10.0
    ax.text(formula_x, 6.8, 'Градиенты (результат обратного прохода):',
            fontsize=12, fontweight='bold')

    grads = [
        (5.8, r'$\frac{\partial L}{\partial w_1} = \delta_1 x_1 + \delta_2 x_2 + \delta_3 x_3$'),
        (4.5, r'$\frac{\partial L}{\partial w_2} = \delta_1 x_2 + \delta_2 x_3 + \delta_3 x_4$'),
        (3.2, r'$\frac{\partial L}{\partial w_3} = \delta_1 x_3 + \delta_2 x_4 + \delta_3 x_5$'),
        (1.8, r'$\frac{\partial L}{\partial b} = \delta_1 + \delta_2 + \delta_3$'),
    ]
    for y, s in grads:
        ax.text(formula_x, y, s, fontsize=12, color='#B9770E',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF9E7',
                          edgecolor='#E67E22', lw=1.5))

    plt.tight_layout()
    plt.show()


draw_backward_pass()



## 1.6. Обновление параметров

Заключительный шаг — собственно обучение. Все четыре параметра обновляются по правилу градиентного спуска с шагом обучения $\eta$. Ключевой момент, который часто упускают: **производные в этом правиле берутся в той самой точке пространства параметров, в которой сеть сейчас находится**, то есть после прямого прохода с текущими весами. Формально, если обозначить текущее состояние параметров через

$$
\theta^{(t)} = \bigl(w_1^{(t)},\, w_2^{(t)},\, w_3^{(t)},\, b^{(t)}\bigr),
$$

а индекс $t$ — номер шага обучения, то

$$
w_m^{(t+1)} := w_m^{(t)} - \eta \left.\frac{\partial L}{\partial w_m}\right|_{\theta^{(t)}}, \qquad b^{(t+1)} := b^{(t)} - \eta \left.\frac{\partial L}{\partial b}\right|_{\theta^{(t)}}.
$$

### Что означает «в точке $\theta^{(t)}$»

Когда мы пишем, например,

$$
\left.\frac{\partial L}{\partial w_1}\right|_{\theta^{(t)}} = \delta_1^{(t)} x_1 + \delta_2^{(t)} x_2 + \delta_3^{(t)} x_3,
$$

все величины в правой части — $\delta_i^{(t)}$ — **вычислены при текущих значениях параметров** $w_1^{(t)}, \dots, b^{(t)}$, то есть взяты из последнего прямого прохода. Поэтому градиентный спуск идёт итеративно:

1. **Прямой проход** с текущими $\theta^{(t)}$ даёт $z_i^{(t)}$, $a_i^{(t)}$.
2. **Обратный проход** с этими величинами даёт $\delta_i^{(t)}$, а затем и все четыре градиента — **в точке $\theta^{(t)}$**.
3. **Шаг обновления** сдвигает параметры в направлении, противоположном градиенту.
4. Цикл повторяется; теперь производные будут браться уже в точке $\theta^{(t+1)}$.

## 1.7. Один полный шаг обучения на числах

Чтобы формулы перестали быть абстрактными, проделаем один шаг алгоритма вручную. Возьмём:

- вход $x = (1, 0, 2, -1, 3)$;
- истинные ответы $t = (1, 0, 1)$;
- начальные параметры $w_1 = 0.5$, $w_2 = -0.5$, $w_3 = 0.25$, $b = 0.1$;
- шаг обучения $\eta = 0.1$;
- $\sigma(z) = 1/(1 + e^{-z})$, $\sigma'(z) = \sigma(z)\,(1-\sigma(z))$.

### Прямой проход

$$
z_1 = 0.1 + 0.5(1) - 0.5(0) + 0.25(2) = 1.1,
$$
$$
z_2 = 0.1 + 0.5(0) - 0.5(2) + 0.25(-1) = -1.15,
$$
$$
z_3 = 0.1 + 0.5(2) - 0.5(-1) + 0.25(3) = 2.35.
$$

$$
a_1 = \sigma(1.1) \approx 0.75026,
$$
$$
a_2 = \sigma(-1.15) \approx 0.24049,
$$
$$
a_3 = \sigma(2.35) \approx 0.91293.
$$

Потеря:

$$
L = \tfrac{1}{2}\bigl[(0.75026 - 1)^2 + (0.24049 - 0)^2 + (0.91293 - 1)^2\bigr] \approx 0.06390.
$$

### Обратный проход

$$
\delta_1 = (0.75026 - 1) \cdot 0.75026(1 - 0.75026) \approx -0.04680,
$$
$$
\delta_2 = (0.24049 - 0) \cdot 0.24049(1 - 0.24049) \approx 0.04393,
$$
$$
\delta_3 = (0.91293 - 1) \cdot 0.91293(1 - 0.91293) \approx -0.00692.
$$

Градиенты:

$$
\frac{\partial L}{\partial b} = -0.04680 + 0.04393 - 0.00692 = -0.00979,
$$
$$
\frac{\partial L}{\partial w_1} = (-0.04680)(1) + (0.04393)(0) + (-0.00692)(2) = -0.06064,
$$
$$
\frac{\partial L}{\partial w_2} = (-0.04680)(0) + (0.04393)(2) + (-0.00692)(-1) = 0.09478,
$$
$$
\frac{\partial L}{\partial w_3} = (-0.04680)(2) + (0.04393)(-1) + (-0.00692)(3) = -0.15829.
$$

### Обновление

$$
w_1 \leftarrow 0.5 - 0.1(-0.06064) = 0.50606,
$$
$$
w_2 \leftarrow -0.5 - 0.1(0.09478) = -0.50948,
$$
$$
w_3 \leftarrow 0.25 - 0.1(-0.15829) = 0.26583,
$$
$$
b \leftarrow 0.1 - 0.1(-0.00979) = 0.10098.
$$

Если теперь выполнить прямой проход с новыми весами, потеря немного уменьшится.

## 1.8. Полная программа

Соберём всё вместе: класс `TinyConv1D` реализует прямой проход, обратный проход и шаг обучения.




In [ ]:
"""
Свёртка 1D из §1: прямой проход, обратный проход, обновление параметров
и три рисунка (Рис. 1.1, 1.2, 1.3).

Исправления по сравнению с исходными фрагментами:
  • все строки с LaTeX-командами — raw (r'...'), SyntaxWarning нет;
  • f-строка для дельт — rf'$\\delta_{i+1}$' с ОДИНАРНЫМИ фигурными скобками
    (в исходнике было {{i+1}} → подставлялся literal "{i+1}");
  • draw_backward_pass показывает и прямой (синие), и обратный (красные
    пунктирные) проход, как обещает §1.5;
  • loss вынесен в метод класса, demo_one_step печатает всё, что в §1.7;
  • sigmoid численно устойчива (без OverflowError);
  • backward не пересчитывает exp повторно — берёт a из прямого прохода.
"""

import math

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch


# ===========================================================================
# 1. Математика
# ===========================================================================
def sigmoid(z: float) -> float:
    """Численно устойчивая логистическая функция."""
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    ez = math.exp(z)            # z < 0 ⇒ ez ∈ (0, 1], переполнения нет
    return ez / (1.0 + ez)


def sigmoid_prime_from_a(a: float) -> float:
    """σ'(z) через уже посчитанное a = σ(z)."""
    return a * (1.0 - a)


# ===========================================================================
# 2. Модель
# ===========================================================================
class TinyConv1D:
    """Свёртка 1D: одно ядро, stride=1, padding=0, сигмоида, half-MSE."""

    def __init__(self, w, b: float):
        self.w = list(w)
        self.b = float(b)

    def __repr__(self) -> str:
        return f'TinyConv1D(w={self.w}, b={self.b:.5f})'

    # --- прямой проход -----------------------------------------------------
    def forward(self, x):
        n, K = len(x), len(self.w)
        if K > n:
            raise ValueError(f'Ядро длиннее входа: K={K} > n={n}')
        o = n - K + 1
        z = [0.0] * o
        a = [0.0] * o
        for i in range(o):
            z[i] = self.b + sum(self.w[m] * x[i + m] for m in range(K))
            a[i] = sigmoid(z[i])
        return z, a

    # --- потеря ------------------------------------------------------------
    @staticmethod
    def loss(a, t):
        """L = ½ Σ (a_i − t_i)²."""
        return 0.5 * sum((ai - ti) ** 2 for ai, ti in zip(a, t))

    # --- обратный проход ---------------------------------------------------
    def backward(self, x, t, z, a):
        """Возвращает (dw, db, delta). δ уже включает σ'(z)."""
        n, K = len(x), len(self.w)
        o = n - K + 1

        delta = [(a[i] - t[i]) * sigmoid_prime_from_a(a[i]) for i in range(o)]
        db = sum(delta)
        dw = [sum(delta[i] * x[i + m] for i in range(o)) for m in range(K)]
        return dw, db, delta

    # --- шаг обучения ------------------------------------------------------
    def update(self, dw, db, eta: float):
        for m in range(len(self.w)):
            self.w[m] -= eta * dw[m]
        self.b -= eta * db


# ===========================================================================
# 3. Рис. 1.1 — архитектура свёртки 1D
# ===========================================================================
def draw_architecture():
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 7)
    ax.axis('off')

    # Вход x1..x5
    for i, val in enumerate(['x₁', 'x₂', 'x₃', 'x₄', 'x₅']):
        ax.add_patch(patches.Rectangle((1 + i * 1.2, 2), 1, 1,
                                       facecolor='#AEDFF7', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(1.5 + i * 1.2, 2.5, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(3.4, 3.6, r'Входной сигнал $x$',
            ha='center', fontsize=12, style='italic')

    # Ядро w1..w3
    for i, val in enumerate(['w₁', 'w₂', 'w₃']):
        ax.add_patch(patches.Rectangle((1 + i * 1.2, 4.5), 1, 1,
                                       facecolor='#FFE29A', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(1.5 + i * 1.2, 5.0, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(2.7, 6.0, r'Ядро (фильтр) $w$',
            ha='center', fontsize=12, style='italic')

    # Стрелки ядро → окно
    for i in range(3):
        ax.annotate('', xy=(1.5 + i * 1.2, 3.05), xytext=(1.5 + i * 1.2, 4.45),
                    arrowprops=dict(arrowstyle='->', color='#B22222', lw=2),
                    zorder=2)

    # Рамка первого окна
    ax.add_patch(patches.Rectangle((0.8, 1.7), 3.6, 1.6, fill=False,
                                   edgecolor='#E74C3C', linestyle='--',
                                   lw=2.5, zorder=5))
    ax.text(2.6, 1.3, 'Окно 1', ha='center', fontsize=11,
            color='#E74C3C', fontweight='bold')

    # Выход z1..z3
    for i, val in enumerate(['z₁', 'z₂', 'z₃']):
        ax.add_patch(patches.Rectangle((9 + i * 1.5, 2), 1, 1,
                                       facecolor='#E8E8E8', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(9.5 + i * 1.5, 2.5, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(10.5, 3.5, r'Выход $z$ (длина 3)',
            ha='center', fontsize=12, style='italic')

    # Скалярное произведение: окно → z1
    ax.add_patch(FancyArrowPatch((4.4, 2.0), (8.9, 2.5),
                                 arrowstyle='-|>', mutation_scale=20,
                                 lw=2.5, color='#2C3E50',
                                 connectionstyle='arc3,rad=-0.2', zorder=2))
    ax.text(6.6, 1.5, 'Скалярное\nпроизведение',
            ha='center', fontsize=11, color='#2C3E50')

    # Смещение b
    ax.annotate('', xy=(9.5, 1.95), xytext=(9.5, 0.5),
                arrowprops=dict(arrowstyle='->', color='#1F6FB2', lw=2.5),
                zorder=2)
    ax.text(9.8, 1.0, r'Смещение $b$',
            fontsize=12, color='#1F6FB2', fontweight='bold')

    # Активация
    ax.add_patch(patches.Rectangle((13.5, 1.0), 1.2, 3.0,
                                   facecolor='#B8E0B8', edgecolor='black',
                                   lw=1.5, zorder=3))
    ax.text(14.1, 2.5, 'Активация\n' + r'$\sigma(z)$',
            ha='center', va='center', fontsize=12, zorder=4)
    ax.annotate('', xy=(13.45, 2.5), xytext=(12.5, 2.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2),
                zorder=2)

    ax.text(7.5, 0.0,
            r'Рис. 1.1. Свёртка 1D: окно из 3 элементов и ядро дают первый выход $z_1$',
            ha='center', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()


# ===========================================================================
# 4. Рис. 1.2 — прямой проход
# ===========================================================================
def draw_forward_pass():
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 8)
    ax.axis('off')

    for i, val in enumerate(['$x_1$', '$x_2$', '$x_3$', '$x_4$', '$x_5$']):
        ax.add_patch(patches.Rectangle((1 + i * 1.2, 3), 1, 1,
                                       facecolor='#AEDFF7', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(1.5 + i * 1.2, 3.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    for i, val in enumerate(['$w_1$', '$w_2$', '$w_3$']):
        ax.add_patch(patches.Rectangle((1 + i * 1.2, 5.5), 1, 1,
                                       facecolor='#FFE29A', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(1.5 + i * 1.2, 6.0, val, ha='center', va='center',
                fontsize=14, zorder=4)

    for i in range(3):
        ax.annotate('', xy=(1.5 + i * 1.2, 4.05), xytext=(1.5 + i * 1.2, 5.45),
                    arrowprops=dict(arrowstyle='->', color='#B22222', lw=2),
                    zorder=2)

    ax.add_patch(patches.Rectangle((0.8, 2.7), 3.6, 1.6, fill=False,
                                   edgecolor='#E74C3C', linestyle='--',
                                   lw=2.5, zorder=5))

    for i, val in enumerate(['$z_1$', '$z_2$', '$z_3$']):
        ax.add_patch(patches.Rectangle((9 + i * 1.5, 3), 1, 1,
                                       facecolor='#E8E8E8', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(9.5 + i * 1.5, 3.5, val, ha='center', va='center',
                fontsize=14, zorder=4)

    ax.add_patch(FancyArrowPatch((4.4, 3.0), (8.9, 3.5),
                                 arrowstyle='-|>', mutation_scale=20,
                                 lw=2.5, color='#2C3E50',
                                 connectionstyle='arc3,rad=-0.2', zorder=2))

    ax.add_patch(patches.Rectangle((13.5, 2.0), 1.2, 3.0,
                                   facecolor='#B8E0B8', edgecolor='black',
                                   lw=1.5, zorder=3))
    ax.text(14.1, 3.5, 'Активация\n' + r'$\sigma(z)$',
            ha='center', va='center', fontsize=12, zorder=4)
    ax.annotate('', xy=(13.45, 3.5), xytext=(12.5, 3.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2),
                zorder=2)

    ax.text(4.5, 1.8, r'$z_1 = b + w_1x_1 + w_2x_2 + w_3x_3$',
            ha='center', fontsize=12, color='#2C3E50')
    ax.text(10.5, 1.8, r'$a_1 = \sigma(z_1)$',
            ha='center', fontsize=12, color='#2C3E50')

    ax.text(8.0, 7.5,
            'Рис. 1.2. Прямой проход: скалярное произведение и активация',
            ha='center', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()


# ===========================================================================
# 5. Рис. 1.3 — прямой (синие →) и обратный (красные ⇢) проход
# ===========================================================================
def draw_backward_pass():
    fig, ax = plt.subplots(figsize=(15, 8))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 9)
    ax.axis('off')

    ax.text(8.0, 8.5,
            'Рис. 1.3. Прямой (синие →) и обратный (красные ⇢) проход',
            ha='center', fontsize=14, fontweight='bold')

    # --- Ядро сверху ---
    for i, val in enumerate(['$w_1$', '$w_2$', '$w_3$']):
        ax.add_patch(patches.Rectangle((2 + i * 1.5, 6.5), 1, 1,
                                       facecolor='#FFE29A', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(2.5 + i * 1.5, 7.0, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(3.5, 7.8, r'Ядро (фильтр) $w$',
            ha='center', fontsize=12, style='italic')

    # --- Вход в центре ---
    for i, val in enumerate(['$x_1$', '$x_2$', '$x_3$', '$x_4$', '$x_5$']):
        ax.add_patch(patches.Rectangle((1 + i * 1.3, 4), 1, 1,
                                       facecolor='#AEDFF7', edgecolor='black',
                                       lw=1.5, zorder=3))
        ax.text(1.5 + i * 1.3, 4.5, val, ha='center', va='center',
                fontsize=14, zorder=4)
    ax.text(3.5, 3.3, r'Входной сигнал $x$',
            ha='center', fontsize=12, style='italic')

    # Прямой проход: w → x (синие сплошные)
    for i in range(3):
        ax.annotate('', xy=(2.5 + i * 1.5, 5.05), xytext=(2.5 + i * 1.5, 6.45),
                    arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2),
                    zorder=2)

    # Рамка первого окна
    ax.add_patch(patches.Rectangle((0.8, 3.7), 4.1, 1.6, fill=False,
                                   edgecolor='#2C3E50', linestyle='--',
                                   lw=2, zorder=5))

    # --- Дельты снизу ---
    for i, (cx, cy) in enumerate([(2.5, 1.5), (4.5, 1.5), (6.5, 1.5)]):
        ax.add_patch(patches.Circle((cx, cy), 0.45,
                                    facecolor='#FADBD8', edgecolor='#E74C3C',
                                    lw=2, zorder=3))
        ax.text(cx, cy, rf'$\delta_{i+1}$',         # ← ОДИНАРНЫЕ { }
                ha='center', va='center', fontsize=14,
                color='#C0392B', zorder=4)
    ax.text(4.5, 0.6, r'Ошибки (дельты) $\delta$',
            ha='center', fontsize=12, style='italic', color='#C0392B')

    # Прямой проход: x → δ (синие сплошные)
    for cx in (2.5, 4.5, 6.5):
        ax.annotate('', xy=(cx, 2.05), xytext=(cx, 3.95),
                    arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2),
                    zorder=2)

    # Обратный проход: δ → w (красные пунктирные)
    for (x0, y0), (x1, y1), rad in [
        ((2.1, 1.5), (2.1, 6.4),  0.2),
        ((2.5, 1.9), (3.1, 6.4),  0.2),
        ((4.5, 1.9), (3.5, 6.4),  0.0),
        ((4.5, 1.9), (4.5, 6.4),  0.0),
        ((6.5, 1.9), (4.9, 6.4), -0.2),
        ((6.9, 1.5), (5.9, 6.4), -0.2),
    ]:
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1),
                                     arrowstyle='-|>', mutation_scale=15,
                                     lw=1.5, color='#E74C3C',
                                     linestyle='--',
                                     connectionstyle=f'arc3,rad={rad}'))

    # --- Формулы градиентов справа ---
    ax.text(10.0, 6.8, 'Градиенты (результат обратного прохода):',
            fontsize=12, fontweight='bold')

    for y, s in [
        (5.8, r'$\frac{\partial L}{\partial w_1} = \delta_1 x_1 + \delta_2 x_2 + \delta_3 x_3$'),
        (4.5, r'$\frac{\partial L}{\partial w_2} = \delta_1 x_2 + \delta_2 x_3 + \delta_3 x_4$'),
        (3.2, r'$\frac{\partial L}{\partial w_3} = \delta_1 x_3 + \delta_2 x_4 + \delta_3 x_5$'),
        (1.8, r'$\frac{\partial L}{\partial b} = \delta_1 + \delta_2 + \delta_3$'),
    ]:
        ax.text(10.0, y, s, fontsize=12, color='#B9770E',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF9E7',
                          edgecolor='#E67E22', lw=1.5))

    plt.tight_layout()
    plt.show()


# ===========================================================================
# 6. Демонстрация §1.7 — один полный шаг обучения
# ===========================================================================
def demo_one_step():
    net = TinyConv1D(w=[0.5, -0.5, 0.25], b=0.1)
    x = [1.0, 0.0, 2.0, -1.0, 3.0]
    t = [1.0, 0.0, 1.0]
    eta = 0.1

    print('=' * 62)
    print('§1.7. Один полный шаг обучения на числах')
    print('=' * 62)
    print(f'  x   = {x}')
    print(f'  t   = {t}')
    print(f'  w⁰  = {net.w},  b⁰ = {net.b},  η = {eta}')
    print()

    z, a = net.forward(x)
    L = net.loss(a, t)
    dw, db, delta = net.backward(x, t, z, a)

    print('Прямой проход:')
    for i in range(len(z)):
        print(f'  z{i+1} = {z[i]:8.5f},  a{i+1} = {a[i]:8.5f}')
    print(f'  L  = {L:.5f}')
    print()

    print('Обратный проход:')
    print(f'  δ      = {[round(d, 5) for d in delta]}')
    print(f'  ∂L/∂b  = {db:8.5f}')
    for m in range(len(dw)):
        print(f'  ∂L/∂w{m+1} = {dw[m]:8.5f}')
    print()

    net.update(dw, db, eta)
    print('После шага:')
    print(f'  w¹ = {[round(w, 5) for w in net.w]}')
    print(f'  b¹ = {net.b:.5f}')
    print()
    print('Сверка с текстом §1.7:')
    print('  z = [1.1, -1.15, 2.35]')
    print('  δ = [-0.04680, 0.04393, -0.00692]')
    print('  w = [0.50606, -0.50948, 0.26583], b = 0.10098')
    print('  L ≈ 0.06390')


# ===========================================================================
# 7. Точка входа
# ===========================================================================
if __name__ == '__main__':
    draw_architecture()
    draw_forward_pass()
    draw_backward_pass()
    demo_one_step()



Если запустить `demo_one_step()`, выводятся ровно те числа, что мы посчитали вручную в §1.7. Это хороший способ убедиться, что ни в одной формуле не потерян множитель: если что-то не сходится с ручным расчётом, ошибка почти наверняка в обратном проходе.

## 1.9. Что мы получили

В этой главе на минимальном примере мы разобрали все три фазы обучения свёрточного слоя:

1. **Прямой проход** — скользящее окно, скалярное произведение с ядром и активация.
2. **Обратный проход** — вычисление $\delta_i$ и **суммирование градиентов по всем позициям**, где побывало ядро (из-за разделения весов).
3. **Обновление** — вычитание усреднённого градиента с шагом $\eta$, причём производные берутся **в текущей точке** $\theta^{(t)}$.

В следующих главах мы добавим **Stride (шаг)** и **Padding (добавление нулей)**, а затем перейдём к двумерным свёрткам (картинкам). Число стрелок на схемах вырастет, но структура останется той же: на каждом шаге мы двигаемся по графу вычислений, перенося ошибку назад через производные и суммируя вклады от всех окон. Формулы этой главы — тот строительный блок, из которого складывается обучение более глубоких свёрточных сетей.